# Ray Tracing and Irradiance Analysis

**WORK IN PROGRESS**

Having modeled the extended LED source, I now turn to ray tracing the rays from this source. The purpose is to understand what degree of uniformity we can expect in the the irradiance after the illuminator. 

The algorithms and lens developed in the previous sections have been incorporated into the `main` module.

In [1]:
import numpy as np
from optiland.coatings import BaseCoatingPolarized
from optiland.jones import JonesPolarizerH, JonesQuarterWaveRetarder
from optiland.rays import PolarizationState, PolarizedRays

from main import Collimator, LED


class LinearPolarizer(BaseCoatingPolarized):
    def __init__(self) -> None:
        super().__init__()
        self.jones = JonesPolarizerH()


class QuarterWavePlate(BaseCoatingPolarized):
    def __init__(self, theta: float = 0.0) -> None:
        super().__init__()
        self.jones = JonesQuarterWaveRetarder(theta=theta)


led = LED()

lens = Collimator()
lens.set_polarization(PolarizationState(is_polarized=False))

# Back vertex of the collimator is the asphere (surface 2). Push the space
# after it out to 5 cm for the polarizer, then add the polarizer and a
# quarter-wave plate 2 cm further on, with its fast axis at 45 deg to the
# polarizer's transmission axis to produce circular polarization. Coatings
# are assigned after add_surface() to avoid the Fresnel-coating callback
# that would otherwise overwrite them.
lens.set_thickness(50.0, surface_number=2)

lens.add_surface(index=3, thickness=20.0, material="air")
lens.surface_group.surfaces[3].interaction_model.coating = LinearPolarizer()

lens.add_surface(index=4, thickness=0.0, material="air")
lens.surface_group.surfaces[4].interaction_model.coating = QuarterWavePlate(theta=np.pi / 4)

In [2]:
num_rays = 100_000

led_rays = led.generate_rays(num_rays=num_rays)

# LED rays are generated in the LED's own local frame (z=0 at the emitter).
# Shift into the collimator's global frame, where the object surface sits
# at the object surface's z position rather than 0.
z_offset = lens.surface_group.positions[0, 0]
rays = PolarizedRays(
    led_rays.x,
    led_rays.y,
    led_rays.z + z_offset,
    led_rays.L,
    led_rays.M,
    led_rays.N,
    led_rays.i,
    led_rays.w,
)

lens.surface_group.trace(rays)
rays.update_intensity(lens.polarization_state)

/home/kmd/src/projects/polarized-led-illuminator/.venv/lib/python3.13/site-packages/optiland/geometries/even_asphere.py:105: RuntimeWarning: invalid value encountered in sqrt
  z = r2 / (self.radius * (1 + be.sqrt(1 - (1 + self.k) * r2 / self.radius**2)))
/home/kmd/src/projects/polarized-led-illuminator/.venv/lib/python3.13/site-packages/optiland/geometries/even_asphere.py:126: RuntimeWarning: invalid value encountered in sqrt
  denom = self.radius * be.sqrt(1 - (1 + self.k) * r2 / self.radius**2)


/home/kmd/src/projects/polarized-led-illuminator/.venv/lib/python3.13/site-packages/optiland/jones.py:110: RuntimeWarning: invalid value encountered in divide
  s = 2 * cos_theta_i / (cos_theta_i + root)
/home/kmd/src/projects/polarized-led-illuminator/.venv/lib/python3.13/site-packages/optiland/jones.py:111: RuntimeWarning: invalid value encountered in divide
  p = 2 * n * cos_theta_i / (n**2 * cos_theta_i + root)
